# Lakehouse & Pipeline Studio Showcase

This notebook demonstrates Jormungandr's flagship data engineering capabilities:
1. **Delta Lake & Parquet ACID Log Inspection**: Parsing transaction commits and table schemas.
2. **Vectorized In-Memory Analytics with DuckDB**: Executing SQL aggregations over Lakehouse files.
3. **Lakehouse Parquet Inspector UI**: Auto-launching the columnar block inspector dialog.
4. **Pipeline Studio Lineage DAG Visualizer**: Auto-launching the cubic-bezier DAG visualizer tool window.

In [1]:
# Cell 1: Delta Lake Transaction Log Inspection
import os
import json

delta_table_dir = os.path.abspath('../03_lakehouse_and_sql/mock_delta_table')
log_dir = os.path.join(delta_table_dir, '_delta_log')
log_files = sorted([f for f in os.listdir(log_dir) if f.endswith('.json')])

print(f'Lakehouse Table: {delta_table_dir}')
print(f'Detected {len(log_files)} ACID commit delta log entries:')
for lf in log_files:
    log_path = os.path.join(log_dir, lf)
    with open(log_path, 'r', encoding='utf-8') as f:
        first_action = json.loads(f.readline())
        print(f'  [Commit] {lf}: actions={list(first_action.keys())}')

In [2]:
# Cell 2: Vectorized In-Memory SQL Querying via DuckDB
import duckdb
import os
from IPython.display import display

conn = duckdb.connect()
customers_csv = os.path.abspath('../02_datasets/customers.csv').replace(os.sep, '/')

query = f'''
    SELECT 
        country,
        COUNT(*) AS total_customers,
        ROUND(AVG(credit_score), 1) AS avg_credit_score,
        ROUND(SUM(annual_spend_usd), 2) AS total_market_spend,
        ROUND(AVG(churn_risk_score) * 100, 1) AS avg_churn_risk_pct
    FROM read_csv_auto('{customers_csv}')
    GROUP BY country
    ORDER BY total_market_spend DESC
    LIMIT 10;
'''
df_lake = conn.execute(query).df()
print('Executed vectorized DuckDB query over lakehouse dataset:')
display(df_lake)

### Programmatic Lakehouse & Pipeline Studio Triggers
Executing the cell below programmatically launches both the **Lakehouse Parquet Inspector** and the **Pipeline Studio Lineage DAG** visualizer.

In [3]:
# Cell 3: Launch Lakehouse Parquet Inspector and Pipeline Lineage DAG
import os
import jormungandr as jm

# 1. Launch Lakehouse Parquet Inspector for mock delta table
delta_table_dir = os.path.abspath('../03_lakehouse_and_sql/mock_delta_table')
jm.show_lakehouse(delta_table_dir)

# 2. Launch Pipeline Lineage DAG in Pipeline Studio
jm.show_pipeline_lineage('lakehouse_customer_hourly_sync')

print('[Jormungandr] Lakehouse Parquet Inspector and Pipeline Lineage DAG visualizer triggered successfully!')